# AGILAB API benchmark in Colab

This notebook demonstrates AGILAB benchmark mode from Google Colab.
It installs AGILAB plus the required core runtime packages from GitHub `main`
without a full checkout and benchmarks the built-in `mycode_project`
across two local execution modes:

- `AGI.PYTHON_MODE`
- `AGI.CYTHON_MODE`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_benchmark.ipynb)


In [ ]:
# Public benchmark install path: install AGILAB plus the required core runtime packages from GitHub main.
# The next cell removes stale /content/agilab source imports if they exist.
!python -m pip uninstall -y agilab agi-cluster agi-node agi-env agi-core >/dev/null 2>&1 || true
!python -m pip install -q uv \
    "agi-core @ git+https://github.com/ThalesGroup/agilab.git@main#subdirectory=src/agilab/core/agi-core" \
    "agi-env @ git+https://github.com/ThalesGroup/agilab.git@main#subdirectory=src/agilab/core/agi-env" \
    "agi-node @ git+https://github.com/ThalesGroup/agilab.git@main#subdirectory=src/agilab/core/agi-node" \
    "agi-cluster @ git+https://github.com/ThalesGroup/agilab.git@main#subdirectory=src/agilab/core/agi-cluster" \
    "agilab @ git+https://github.com/ThalesGroup/agilab.git@main"


In [ ]:
import importlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

for bad_prefix in ("/content/agilab/src", "/content/agilab"):
    sys.path = [p for p in sys.path if p != bad_prefix and not p.startswith(bad_prefix + "/")]
for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.")):
        del sys.modules[name]
importlib.invalidate_caches()

import agilab
from agi_cluster.agi_distributor import AGI
from agi_env import AgiEnv

APPS_PATH = Path(agilab.__file__).resolve().parent / "apps"
BUILTIN_ROOT = APPS_PATH / "builtin"
APP = "mycode_project"

def worker_env_ready(app_env):
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if not worker_venv.exists():
        return False
    cmd = ["uv", "--quiet", "run", "--no-sync", "--project", str(worker_venv.parent)]
    pyvers_worker = getattr(app_env, "pyvers_worker", None)
    if pyvers_worker:
        cmd.extend(["--python", str(pyvers_worker)])
    cmd.extend(["python", "-c", "import agi_env, agi_node"])
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    return result.returncode == 0

async def install_if_needed(app_env, *, scheduler="127.0.0.1", workers=None, modes_enabled=0):
    if workers is None:
        workers = {"127.0.0.1": 1}
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if worker_env_ready(app_env):
        return False
    action = "Installing"
    if worker_venv.parent.exists():
        shutil.rmtree(worker_venv.parent, ignore_errors=True)
        action = "Reinstalling"
    print(f"{action} worker for {app_env.app}...")
    await AGI.install(app_env, scheduler=scheduler, workers=workers, modes_enabled=modes_enabled)
    return True

app_env = AgiEnv(apps_path=APPS_PATH, app=APP, verbose=0)

print("Installed apps path:", APPS_PATH)
print("Builtin root:", BUILTIN_ROOT)
print("Benchmark file:", app_env.benchmark)


In [ ]:
await install_if_needed(app_env, scheduler="127.0.0.1", workers={"127.0.0.1": 1}, modes_enabled=0)
benchmark_json = await AGI.run(
    app_env,
    scheduler="127.0.0.1",
    workers={"127.0.0.1": 1},
    mode=[AGI.PYTHON_MODE, AGI.CYTHON_MODE],
)
benchmark_json
